In [1]:
!wget http://files.grouplens.org/datasets/movielens/ml-20m.zip

# Unzip the dataset
!unzip ml-20m.zip

--2025-03-17 20:23:43--  http://files.grouplens.org/datasets/movielens/ml-20m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.65.152
Connecting to files.grouplens.org (files.grouplens.org)|128.101.65.152|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 198702078 (189M) [application/zip]
Saving to: ‘ml-20m.zip’

ml-20m.zip          100%[===================>] 189.50M  84.3MB/s    in 2.2s    

2025-03-17 20:23:45 (84.3 MB/s) - ‘ml-20m.zip’ saved [198702078/198702078]

Archive:  ml-20m.zip
   creating: ml-20m/
  inflating: ml-20m/genome-scores.csv  
  inflating: ml-20m/genome-tags.csv  
  inflating: ml-20m/links.csv        
  inflating: ml-20m/movies.csv       
  inflating: ml-20m/ratings.csv      
  inflating: ml-20m/README.txt       
  inflating: ml-20m/tags.csv         


### Code Explanation

In this code cell, we are performing the following tasks:

1. **Importing Required Libraries:**
   - `pandas`: Used for data manipulation and handling tabular data.
   - `numpy`: Used for numerical operations, specifically for working with arrays.
   - `TfidfVectorizer`: A tool from `sklearn` to convert text data into numerical format based on term frequency-inverse document frequency (TF-IDF).
   - `TruncatedSVD` and `NMF`: Dimensionality reduction techniques used for feature extraction, commonly used in text mining and recommendation systems.
   - `cosine_similarity`: A metric used to calculate the cosine similarity between vectors (e.g., movie content or user ratings).




In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.metrics.pairwise import cosine_similarity

 **Reading the Data:**
   - `movies_metadata`: Loads the movie metadata from the `movies.csv` file. We limit the rows to 100,000 for quicker processing.
   - `ratings`: Loads the user ratings data from the `ratings.csv` file. The rows are again limited to 100,000 for efficient handling.
   - `tags`: Loads the tags (or keywords) associated with movies from the `tags.csv` file, limited to 100,000 rows.

By loading these datasets, we are preparing the data to perform operations such as content-based filtering and collaborative filtering in a movie recommendation system.

In [3]:
movies_metadata = pd.read_csv('ml-20m/movies.csv', nrows=100000)
ratings = pd.read_csv('ml-20m/ratings.csv', nrows=100000)
tags = pd.read_csv('ml-20m/tags.csv', nrows=100000)


In [4]:
movies_metadata.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy




1. **Creating a New 'metadata' Column in `movies_metadata`:**
   - We combine the `genres` column from the `movies_metadata` DataFrame with the `tag` column from the `tags` DataFrame to create a new `metadata` column.
   - The `tag` values from the `tags` DataFrame are first filled with an empty string (`''`) for any missing values (`NaN`), using the `fillna('')` method.
   - The `genres` and `tag` columns are concatenated using the `+` operator to create a single string containing both the genres and the associated tags for each movie. This new string represents the metadata of the movie.

This new `metadata` column will be useful for further processing in content-based filtering, where movie metadata (genres and tags) can help recommend similar movies based on content similarity.


In [5]:
movies_metadata['metadata'] = movies_metadata['genres'] + ' ' + tags['tag'].fillna('')




In this code cell, we perform the following steps to create a user-movie matrix:

1. **Retrieve All Movie IDs from the Complete Dataset:**
   - We first read the `movies.csv` file from the dataset and extract the unique `movieId` values using the `unique()` function. This ensures that we have a list of all available movie IDs in the complete dataset.

2. **Create a User-Movie Matrix from the Sample Data:**
   - Using the `ratings.csv` file, we create a user-movie matrix by using the `pivot()` function. The matrix is constructed with `movieId` as the index (rows), `userId` as the columns, and `rating` as the values. Missing ratings (NaN) are filled with zeros using the `fillna(0)` method.

3. **Reindex the Matrix to Include All Movie IDs:**
   - We reindex the `user_movie_matrix` to include all the movie IDs from the complete dataset, ensuring that every movie in the complete dataset is represented. If any movie ID is missing in the original `user_movie_matrix`, it will be added and filled with zero values (no ratings for those movies).

4. **Check the Shape of the Matrix:**
   - Finally, we print the shape of the `user_movie_matrix` to confirm that the matrix now includes all the rows for every movie in the dataset.




In [6]:
# Assuming you have a full list of movie IDs from the complete dataset
all_movie_ids = pd.read_csv('ml-20m/movies.csv')['movieId'].unique()

# Create the user-movie matrix from the sample
user_movie_matrix = ratings.pivot(index='movieId', columns='userId', values='rating').fillna(0)

# Reindex the matrix to include all movie IDs from the complete dataset, filling missing values with 0
user_movie_matrix = user_movie_matrix.reindex(all_movie_ids, fill_value=0)

# Now user_movie_matrix should have the correct number of rows
print(user_movie_matrix.shape)

(27278, 702)


In [7]:
user_movie_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,693,694,695,696,697,698,699,700,701,702
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,4.0,0.0,0.0,5.0,0.0,4.0,0.0,4.0,...,0.0,4.5,0.0,0.0,0.0,0.0,4.0,4.0,0.0,3.5
2,3.5,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.5,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0
3,0.0,4.0,0.0,0.0,0.0,3.0,3.0,5.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.5,0.0,0.0,3.5,0.0,2.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
131256,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
131258,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0




In this code cell, we perform the following steps to process the movie metadata using TF-IDF and apply dimensionality reduction using TruncatedSVD:

1. **Initialize TF-IDF Vectorizer:**
   - We create an instance of `TfidfVectorizer` with the argument `stop_words='english'`, which removes common English stop words (e.g., 'the', 'and', etc.) during the tokenization process. This helps focus on the meaningful words in the metadata.

2. **Transform Movie Metadata Using TF-IDF:**
   - We apply the `fit_transform()` method on the `metadata` column of the `movies_metadata` DataFrame. This converts the textual metadata into a sparse matrix of TF-IDF features, where each feature corresponds to a term in the movie metadata. The resulting matrix represents the importance of each term relative to the other terms across all movie descriptions.

3. **Convert TF-IDF Matrix to a DataFrame:**
   - We convert the sparse TF-IDF matrix (`metadata_tfidf`) to a dense NumPy array using the `toarray()` method and then create a DataFrame (`tfidf_df`). The index of the DataFrame is set to the original index of the `movies_metadata` DataFrame, ensuring that we can track the movies associated with each row.

4. **Apply TruncatedSVD for Dimensionality Reduction:**
   - We initialize a `TruncatedSVD` model with `n_components=49`, which reduces the dimensionality of the TF-IDF matrix to 49 components. This is a technique to capture the most significant patterns (topics or features) in the metadata while reducing the number of dimensions in the data. This step is used to improve the computational efficiency and avoid overfitting.

5. **Transform the TF-IDF Matrix Using SVD:**
   - We apply the `fit_transform()` method of the `TruncatedSVD` model on the `metadata_tfidf` matrix to reduce its dimensionality, resulting in a lower-dimensional representation (`metadata_svd`) of the movie metadata.


In [8]:
tfidf = TfidfVectorizer(stop_words='english')
metadata_tfidf = tfidf.fit_transform(movies_metadata['metadata'])
tfidf_df = pd.DataFrame(metadata_tfidf.toarray(), index=movies_metadata.index.tolist())
svd = TruncatedSVD(n_components=49, random_state=42)
metadata_svd = svd.fit_transform(metadata_tfidf)

In [9]:
n = 50
latent_matrix_1_df = pd.DataFrame(metadata_svd[:,0:n], index=movies_metadata.title.tolist())


In [10]:

svd = TruncatedSVD(n_components=200)
latent_matrix_2 = svd.fit_transform(user_movie_matrix)
latent_matrix_2_df = pd.DataFrame(
                             latent_matrix_2, index=movies_metadata.title.tolist())

In [11]:
latent_matrix_2_df

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
Toy Story (1995),40.311089,7.732360,-8.529027,-6.559547,-0.517900,-12.040695,-2.181887,-7.117325,-2.207313,-2.317201,...,-3.218978,-2.574067,-1.837308,-2.427318,-0.588688,1.886660,-0.108480,2.228496,1.121945,0.884604
Jumanji (1995),17.829010,10.108802,-6.608959,-1.004083,3.640651,-3.877263,0.354441,-0.969136,-0.389128,-2.546228,...,-0.825229,0.999859,0.887234,-0.478656,1.788363,-0.446319,-1.403486,0.628276,-0.828800,0.019985
Grumpier Old Men (1995),9.024620,7.138243,-3.030521,0.609531,4.274814,-0.541927,-1.011345,-1.460967,-0.470992,0.176217,...,0.232450,-0.231907,-0.666752,-0.302159,-0.706859,0.311700,0.604435,-1.189207,0.760854,0.434608
Waiting to Exhale (1995),1.834689,3.263111,0.461236,-0.580461,2.546131,0.556450,2.306675,-0.303879,-0.935351,-0.024339,...,-0.871104,0.052444,-0.327756,0.431627,0.400730,0.244049,-0.072126,0.556461,-0.442405,-0.220270
Father of the Bride Part II (1995),8.640309,9.214961,-3.369466,-0.400534,4.340913,-0.020033,-1.637575,-1.043688,-0.832471,-1.118795,...,-0.132259,-1.029229,-0.943553,0.345860,-0.637875,0.985437,0.871170,-2.353267,-0.052258,-0.386884
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Kein Bund für's Leben (2007),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"Feuer, Eis & Dosenbier (2002)",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
The Pirates (2014),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Rentun Ruusu (2001),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Function: `recommend_similar_movies(title)`

This function recommends movies similar to a given movie title by using a hybrid recommendation approach that combines both **content-based filtering** and **collaborative filtering**.

#### **Step-by-step breakdown:**

1. **Extract Latent Vectors for the Selected Movie:**
   - `a_1 = np.array(latent_matrix_1_df.loc[title]).reshape(1, -1)`:  
     Retrieves the latent vector representation of the given movie from the **content-based** latent matrix.
   - `a_2 = np.array(latent_matrix_2_df.loc[title]).reshape(1, -1)`:  
     Retrieves the latent vector representation of the given movie from the **collaborative filtering** latent matrix.

2. **Compute Cosine Similarities:**
   - `score_1 = cosine_similarity(latent_matrix_1_df, a_1).reshape(-1)`:  
     Calculates the cosine similarity between the given movie and all other movies in the content-based latent matrix.
   - `score_2 = cosine_similarity(latent_matrix_2_df, a_2).reshape(-1)`:  
     Calculates the cosine similarity between the given movie and all other movies in the collaborative filtering latent matrix.

3. **Combine the Two Similarity Scores:**
   - `hybrid = ((score_1 + score_2) / 2.0)`:  
     Averages the similarity scores from both methods to create a **hybrid recommendation score**.

4. **Create a DataFrame of Similar Movies:**
   - `dictDf = {'content': score_1, 'collaborative': score_2, 'hybrid': hybrid}`:  
     Stores the similarity scores from content-based filtering, collaborative filtering, and their hybrid combination.
   - `similar = pd.DataFrame(dictDf, index=latent_matrix_2_df.index)`:  
     Creates a DataFrame where each movie is associated with its similarity scores.

5. **Sort Movies by Similarity Score:**
   - `similar.sort_values('hybrid', ascending=False, inplace=True)`:  
     Sorts the DataFrame based on the hybrid similarity score, ensuring the most similar movies appear first.

6. **Display the Top 10 Most Similar Movies:**
   - `print(similar[1:].head(11))`:  
     Prints the top 10 recommended movies, excluding the first result (which is the input movie itself).




In [12]:
def recommend_similar_movies(title):
    # take the latent vectors for a selected movie from both content
    # and collaborative matrixes
    a_1 = np.array(latent_matrix_1_df.loc[title]).reshape(1, -1)
    a_2 = np.array(latent_matrix_2_df.loc[title]).reshape(1, -1)

    # calculate the similartity of this movie with the others in the list
    score_1 = cosine_similarity(latent_matrix_1_df, a_1).reshape(-1)
    score_2 = cosine_similarity(latent_matrix_2_df, a_2).reshape(-1)

    # an average measure of both content and collaborative
    hybrid = ((score_1 + score_2)/2.0)

    # form a data frame of similar movies
    dictDf = {'content': score_1 , 'collaborative': score_2, 'hybrid': hybrid}
    similar = pd.DataFrame(dictDf, index = latent_matrix_2_df.index )

    #sort it on the basis of either: content, collaborative or hybrid
    similar.sort_values('hybrid', ascending=False, inplace=True)

    print(similar[1:].head(11))

In [13]:
recommend_similar_movies("Toy Story 2 (1999)")

                                   content  collaborative    hybrid
Toy Story (1995)                  0.999697       0.611296  0.805496
Monsters, Inc. (2001)             0.999869       0.590275  0.795072
Bug's Life, A (1998)              0.879233       0.614660  0.746946
Who Framed Roger Rabbit? (1988)   0.849926       0.627726  0.738826
Toy Story 3 (2010)                0.903082       0.424394  0.663738
Emperor's New Groove, The (2000)  0.999420       0.310271  0.654846
Shrek (2001)                      0.759825       0.540336  0.650081
Ice Age (2002)                    0.837607       0.441942  0.639775
Finding Nemo (2003)               0.655887       0.606195  0.631041
Antz (1998)                       0.817830       0.436257  0.627043
Shrek the Third (2007)            0.999862       0.249807  0.624834


### Function: `popularity_recommender(movies_metadata, ratings, top_n=10)`

This function recommends the top *n* popular movies based on the number of ratings they have received. It calculates the popularity score for each movie and returns the most popular ones.

#### **Step-by-step breakdown:**

1. **Calculate the Popularity Score:**
   - `movie_popularity = ratings.groupby('movieId').size().reset_index(name='rating_count')`:  
     Groups the `ratings` DataFrame by `movieId` and counts the number of ratings each movie has received, storing this count as `rating_count`.

2. **Merge Popularity with Movie Metadata:**
   - `movie_popularity = movie_popularity.merge(movies_metadata, on='movieId')`:  
     Merges the `movie_popularity` DataFrame (which now contains the movie rating counts) with the `movies_metadata` DataFrame to get additional movie details like the title.

3. **Sort Movies by Popularity:**
   - `popular_movies = movie_popularity.sort_values(by='rating_count', ascending=False)`:  
     Sorts the movies based on the `rating_count` in descending order, so that the most popular movies appear first.

4. **Return the Top *n* Popular Movies:**
   - `return popular_movies.head(top_n)`:  
     Returns the top `n` most popular movies based on the sorted `rating_count`.





In [14]:
def popularity_recommender(movies_metadata, ratings, top_n=10):
    # Compute the popularity score based on the number of ratings
    movie_popularity = ratings.groupby('movieId').size().reset_index(name='rating_count')
    movie_popularity = movie_popularity.merge(movies_metadata, on='movieId')
    popular_movies = movie_popularity.sort_values(by='rating_count', ascending=False)
    return popular_movies.head(top_n)

# Get top 10 popular movies
popular_movies = popularity_recommender(movies_metadata, ratings)
print(popular_movies[['movieId', 'title', 'rating_count']])


      movieId                                      title  rating_count
267       296                        Pulp Fiction (1994)           350
323       356                        Forrest Gump (1994)           340
286       318           Shawshank Redemption, The (1994)           305
436       480                       Jurassic Park (1993)           302
538       593           Silence of the Lambs, The (1991)           295
235       260  Star Wars: Episode IV - A New Hope (1977)           264
101       110                          Braveheart (1995)           262
534       589          Terminator 2: Judgment Day (1991)           256
2106     2571                         Matrix, The (1999)           253
482       527                    Schindler's List (1993)           247


 Matrix Factorization Recommender

### Function: `matrix_factorization_recommender(user_id, user_movie_matrix, top_n=10)`

This function recommends movies to a specific user using matrix factorization with Singular Value Decomposition (SVD). It predicts the user's preferences based on the user-movie interaction matrix.

#### **Step-by-step breakdown:**

1. **Matrix Decomposition (SVD):**
   - `U, sigma, Vt = np.linalg.svd(user_movie_matrix, full_matrices=False)`:  
     Decomposes the user-movie matrix into three components:  
     - `U`: The user-feature matrix  
     - `sigma`: The singular value matrix  
     - `Vt`: The movie-feature matrix  
     `full_matrices=False` ensures that the decomposition uses reduced dimensions for efficiency.

2. **Get User Profile in the Latent Space:**
   - `user_profile = np.dot(np.dot(U[user_id-1, :], sigma), Vt)`:  
     Calculates the user's profile by multiplying the user's latent factors (`U`) with the singular values (`sigma`) and the movie features (`Vt`). This gives a vector of predicted ratings for all movies.

3. **Predict Movie Scores:**
   - `movie_scores = user_profile`:  
     The movie scores (predicted ratings) are taken directly from the user's profile.

4. **Create a DataFrame of Recommendations:**
   - `recommendations = pd.DataFrame({'movieId': user_movie_matrix.columns, 'rating': movie_scores})`:  
     Creates a DataFrame with the `movieId` and the corresponding predicted `rating`.

5. **Filter Already Rated Movies:**
   - `recommendations = recommendations[~recommendations['movieId'].isin(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index)]`:  
     Filters out movies that the user has already rated from the recommendations.

6. **Sort and Return the Top *n* Movies:**
   - `return recommendations.sort_values(by='rating', ascending=False).head(top_n)`:  
     Sorts the recommendations by the predicted rating in descending order and returns the top `n` recommended movies.




In [15]:
def matrix_factorization_recommender(user_id, user_movie_matrix, top_n=10):
    # Decompose the user-movie matrix using SVD
    U, sigma, Vt = np.linalg.svd(user_movie_matrix, full_matrices=False)
    sigma = np.diag(sigma)

    # Get the user's profile in the latent space
    user_profile = np.dot(np.dot(U[user_id-1, :], sigma), Vt)

    # Get the movie scores from the user's profile
    movie_scores = user_profile

    # Create a DataFrame with movieId and predicted ratings
    recommendations = pd.DataFrame({'movieId': user_movie_matrix.columns, 'rating': movie_scores})
    recommendations = recommendations[~recommendations['movieId'].isin(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index)]

    return recommendations.sort_values(by='rating', ascending=False).head(top_n)
# Example usage
user_id = 1  # Replace with a valid user_id from your dataset
matrix_factorization_recommendations = matrix_factorization_recommender(user_id, user_movie_matrix)
print("Matrix Factorization Recommendations:")
print(matrix_factorization_recommendations[['movieId', 'rating']])

Matrix Factorization Recommendations:
     movieId        rating
1          2  3.271960e-14
8          9  1.363200e-14
102      103  8.786632e-15
341      342  8.594471e-15
503      504  8.572272e-15
539      540  8.502205e-15
441      442  8.439213e-15
450      451  8.010140e-15
433      434  7.930207e-15
447      448  7.625302e-15
